In [3]:
import pandas as pd
import numpy as np
import joblib
import hashlib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error

import mlflow
import mlflow.sklearn

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("warehouse_pick_prediction")

2026/11/28 14:11:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/28 14:11:41 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

<Experiment: artifact_location='file:c:/Users/princ/Music/G099_MP215/models/mlruns/1', creation_time=1795855303801, experiment_id='1', last_update_time=1795855303801, lifecycle_stage='active', name='warehouse_pick_prediction', tags={}>

In [5]:
def calculate_sha256(filepath):

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as f:

        for block in iter(lambda: f.read(4096), b""):
            sha256.update(block)

    return sha256.hexdigest()

In [11]:
TRAIN_PATH = r"C:\Users\princ\Music\G099_MP215\data\processed\train_clean.csv"

df = pd.read_csv(TRAIN_PATH)

df.head()

,record_id,date,items_in_order,aisle_distance_m,picker_exp_months,shelf_height_m,shift_hour,order_weight_kg,zone_type,shift,pick_seconds
0,R00001,2026-01-01,10,301,22.0,1.34,16,2.57,bulky,Night,175
1,R00002,2026-01-01,29,377,1.0,1.33,5,21.46,chilled,Morning,484
2,R00003,2026-01-01,22,192,25.0,1.78,21,19.12,chilled,Night,307
3,R00004,2026-01-01,2,222,17.0,1.36,22,1.77,bulky,Evening,146
4,R00005,2026-01-01,24,320,59.0,1.23,11,22.84,bulky,Evening,267


In [12]:
FEATURES = [
    "items_in_order",
    "aisle_distance_m",
    "picker_exp_months",
    "shelf_height_m",
    "shift_hour",
    "order_weight_kg",
    "zone_type",
    "shift"
]

TARGET = "pick_seconds"

In [13]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
num_cols = [
    "items_in_order",
    "aisle_distance_m",
    "picker_exp_months",
    "shelf_height_m",
    "shift_hour",
    "order_weight_kg"
]

cat_cols = [
    "zone_type",
    "shift"
]

preprocessor = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        num_cols
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        cat_cols
    )
])

In [15]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    )
}

best_mae = float("inf")
best_model = None
best_name = None

data_hash = calculate_sha256(TRAIN_PATH)

In [16]:
for model_name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    with mlflow.start_run(run_name=model_name):

        pipe.fit(X_train, y_train)

        preds = pipe.predict(X_test)

        mae = mean_absolute_error(
            y_test,
            preds
        )

        mlflow.log_param(
            "model",
            model_name
        )

        mlflow.log_param(
            "data_sha256",
            data_hash
        )

        mlflow.log_metric(
            "mae",
            mae
        )

        mlflow.sklearn.log_model(
            pipe,
            "model"
        )

        print(model_name, mae)

        if mae < best_mae:

            best_mae = mae
            best_model = pipe
            best_name = model_name

2026/11/28 14:15:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LinearRegression 14.6062070049609


2026/11/28 14:15:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RandomForest 21.06995


2026/11/28 14:16:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


GradientBoosting 16.12873160093355


In [20]:
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/model.joblib"
)

['models/model.joblib']

In [21]:
joblib.dump(
    best_model,
    "models/model.joblib"
)

print("Best Model:", best_name)
print("Best MAE:", best_mae)

Best Model: LinearRegression
Best MAE: 14.6062070049609
